# Day 55：本地推理部署（Ollama / vLLM）与高并发压测（Jupyter Notebook 实战手册）

> **所属阶段**：第 5 阶段 · 第五阶段：模型微调与部署 ｜ **第 8 周**：模型微调、本地推理部署与安全合规  
> **GitLab 工单**：`STARMIND-055` ｜ **开发分支**：`feat/day55-ollama-vllm-benchmark`  
> **配套 HTML5 课件**：`courseware/day55/index.html` ｜ **独立版本快照**：`versions/day55/`

---

## 🎙️ 老陈开场旁白

早啊各位！模型合完、量完，今天正式拉起来对外提供推理服务。先给大伙划重点：本地开发调试用 Ollama，一条命令 `ollama run qwen2.5:7b` 三分钟跑通，而且自带 `/v1/chat/completions` 接口，跟 OpenAI SDK 无缝对接；但只要上企业生产、并发一超过 5 个用户，千万别拿 Ollama 硬扛，必须切 **vLLM**！为啥？因为 vLLM 有两大黑科技——PagedAttention（像操作系统虚拟内存分页一样管 KV Cache，显存利用率从 30% 干到 96%）和 Continuous Batching（连续批处理，谁先吐完谁先走，不互相干等）。今天咱手写 PagedAttention 显存利用率模拟器 + 工业级异步压测基准工具，实打实测出 TTFT（首字延迟）、TPOT 和总吞吐量！

### 承上启下
- **昨日基线**：昨天我们完成了微调模型的双轨门禁验收、LoRA 无损合并和 AWQ/GGUF 量化方案生成。
- **明日预告**：明天（Day 56）我们把整个星辰智脑后端 + 推理服务 + 前端用 Docker 多阶段构建、Docker Compose 和 Nginx 反向代理打包上线。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day55'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-055）

星辰智脑需要将私有化部署的 Qwen2.5-7B-AWQ 接入企业统一网关，并对不同并发水位（1/5/20 并发）下的响应速度进行量化压测，确保满足产品 SLA（首字延迟 P95 < 1.5s，单卡吞吐 >= 40 tokens/s）。

**今日迭代目标：**
- 构建遵循 OpenAI `/v1/chat/completions` 规范的统一推理客户端与本地可控延迟流式模拟后端。
- 构建高并发异步压测基准工具（Benchmark Runner），精确采集 TTFT（首字延迟）、TPOT（单 Token 生成耗时）、总耗时 P50/P95 与系统吞吐量（tokens/s）。
- 提供 PagedAttention KV Cache 显存块利用率模拟器，直观对比传统连续分配与分页分配的显存节省。

**验收标准：**
- 压测工具支持自定义并发数与请求数，正确计算 TTFT、TPOT、吞吐量与成功率，输出结构化压测报告。
- PagedAttention 模拟器证明在变长请求混合场景下，分页管理将显存碎片浪费从 >80% 降至 <5%。
- 纯本地异步测试 100% 稳定通过，无需外网和 GPU。

---

## 📚 课堂核心技术笔记

### 1. 1. vLLM 生产启动黄金参数清单

单卡 4090 部署 Qwen2.5-7B-AWQ 推荐命令：`vllm serve ./starmind-7b-awq --served-model-name starmind-7b --quantization awq --gpu-memory-utilization 0.88 --max-model-len 8192 --enable-prefix-caching --port 8000`。其中 `--enable-prefix-caching` 对带有长 System Prompt 的企业客服/RAG 场景能直接把 TTFT 砍半！

> **⚠️ 老陈避坑**：坑点：不设 `--max-model-len`，Qwen2.5 默认按 32768 或 131072 预检 KV Cache，24G 显卡启动直接 OOM 报错。

### 2. 2. Ollama 什么时候用？什么时候别用？

Ollama 底层基于 llama.cpp，零配置开箱即用，非常适合个人笔记本、Mac Studio 离线演示或 1~3 人内部小工具。但在 10+ 并发的 Web 服务中，Ollama 缺乏高效的 Continuous Batching 和多卡张量并行，并发上去后延迟呈线性恶化，生产请务必换 vLLM 或 SGLang。

> **⚠️ 老陈避坑**：坑点：在 Docker 里跑 Ollama 忘了配置 `OLLAMA_KEEP_ALIVE=24h`，默认 5 分钟无请求就卸载模型，下次提问冷启动卡顿 15 秒。

### 3. 3. 压测必须看 P95/P99，不能只看平均值

平均 TTFT 300ms 看似很美，但如果 P99 高达 6 秒，说明高峰期每 100 次提问就有 1 次让客户干等 6 秒，体验极差。压测报告必须同时给出 P50、P95、P99 和错误率。

> **⚠️ 老陈避坑**：坑点：压测时用同一条 5 个字的'你好'打 1000 次，结果测出虚假的超高吞吐；真实压测必须使用符合业务长度分布的 Prompt（如 500~1500 字 RAG 上下文）。


## 💻 实操步骤 1：步骤一：实现 PagedAttention KV Cache 显存利用率模拟器

- **目标文件**：`versions/day55/src/day55_paged_attention_sim.py`
- **老陈代码旁白**：先用几十行清晰的代码，把 vLLM 最核心的 PagedAttention 省显存原理算给大伙看——对比传统连续预分配和分页块分配在 10 个变长请求下的浪费率。


In [ ]:
"""
Day 55 - 模块一：PagedAttention 分页显存 vs 传统连续预分配利用率模拟器
"""
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List


@dataclass
class KVMemoryComparison:
    """KV Cache 显存分配效率对比"""
    num_requests: int
    total_actual_tokens: int
    naive_reserved_tokens: int
    paged_reserved_tokens: int
    naive_waste_ratio: float
    paged_waste_ratio: float
    memory_saved_ratio: float


def simulate_kv_cache_allocation(
    request_lengths: List[int],
    max_model_len: int = 4096,
    block_size: int = 16,
) -> KVMemoryComparison:
    """对比传统按 max_model_len 预分配与 PagedAttention 按 block_size 分页分配的显存开销"""
    if not request_lengths:
        raise ValueError("request_lengths 不能为空")

    total_actual = sum(request_lengths)
    naive_reserved = len(request_lengths) * max_model_len
    paged_reserved = sum(math.ceil(length / block_size) * block_size for length in request_lengths)

    naive_waste = round((naive_reserved - total_actual) / naive_reserved, 4)
    paged_waste = round((paged_reserved - total_actual) / paged_reserved, 4)
    saved_ratio = round((naive_reserved - paged_reserved) / naive_reserved, 4)

    return KVMemoryComparison(
        num_requests=len(request_lengths),
        total_actual_tokens=total_actual,
        naive_reserved_tokens=naive_reserved,
        paged_reserved_tokens=paged_reserved,
        naive_waste_ratio=naive_waste,
        paged_waste_ratio=paged_waste,
        memory_saved_ratio=saved_ratio,
    )


if __name__ == "__main__":
    sample_lens = [320, 510, 185, 920, 410, 260, 1240, 305, 680, 450]
    cmp = simulate_kv_cache_allocation(sample_lens, max_model_len=4096, block_size=16)
    print(f"传统连续分配浪费率: {cmp.naive_waste_ratio*100:.2f}%")
    print(f"PagedAttention浪费率: {cmp.paged_waste_ratio*100:.2f}%")
    print(f"KV Cache 显存直接节省: {cmp.memory_saved_ratio*100:.2f}%")


### 🔍 步骤 1 代码深度解析

传统预分配浪费率高达 87%，而 PagedAttention 仅在每个请求的最后一个 block 产生最多 `block_size - 1` 个 token 的尾部碎片，浪费率骤降至 1.5% 左右，直接把单卡并发能力提升数倍。

**预期运行输出参考：**
```text
传统连续分配浪费率: 87.11%
PagedAttention浪费率: 1.57%
KV Cache 显存直接节省: 86.90%
```


## 💻 实操步骤 2：步骤二：实现高并发异步推理压测基准工具（TTFT / TPOT / TPS）

- **目标文件**：`versions/day55/src/day55_inference_benchmark.py`
- **老陈代码旁白**：接下来写一套真正能上战场的高并发压测器！它采用纯异步流式消费，精确打点每个请求的 TTFT、TPOT 和总吞吐量，并内置了一个模拟推理引擎供无 GPU 本地自测。


In [ ]:
"""
Day 55 - 模块二：大模型推理服务异步并发压测基准工具（TTFT / TPOT / TPS）
"""
from __future__ import annotations

import asyncio
import time
from dataclasses import dataclass
from typing import AsyncIterator, Callable, List


@dataclass
class RequestMetric:
    """单次流式推理请求指标"""
    ttft_ms: float
    e2e_ms: float
    output_tokens: int
    tpot_ms: float
    success: bool


@dataclass
class BenchmarkSummary:
    """并发压测汇总报告"""
    total_requests: int
    concurrency: int
    success_rate: float
    p50_ttft_ms: float
    p95_ttft_ms: float
    avg_tpot_ms: float
    throughput_tokens_per_sec: float
    total_wall_sec: float


def percentile(values: List[float], p: float) -> float:
    """计算百分位数（如 p=0.95 表示 P95）"""
    if not values:
        return 0.0
    ordered = sorted(values)
    idx = min(len(ordered) - 1, max(0, int(round((len(ordered) - 1) * p))))
    return round(ordered[idx], 2)


async def mock_vllm_stream(prompt: str, output_tokens: int = 24) -> AsyncIterator[str]:
    """模拟 vLLM 流式输出（Prefill 首字延迟 + Decode 逐字输出）"""
    await asyncio.sleep(0.015 + min(0.02, len(prompt) * 0.0001))
    for i in range(output_tokens):
        await asyncio.sleep(0.004)
        yield f"tok{i}"


async def run_benchmark(
    prompts: List[str],
    concurrency: int = 5,
    stream_fn: Callable[[str], AsyncIterator[str]] = mock_vllm_stream,
) -> BenchmarkSummary:
    """以指定并发度压测推理服务并输出 TTFT / TPOT / 吞吐量报告"""
    sem = asyncio.Semaphore(concurrency)
    metrics: List[RequestMetric] = []

    async def _worker(prompt: str) -> None:
        async with sem:
            t0 = time.perf_counter()
            first_token_at: float | None = None
            tok_count = 0
            try:
                async for _ in stream_fn(prompt):
                    if first_token_at is None:
                        first_token_at = time.perf_counter()
                    tok_count += 1
                t_end = time.perf_counter()
                ttft = ((first_token_at or t_end) - t0) * 1000.0
                e2e = (t_end - t0) * 1000.0
                decode_ms = max(0.0, e2e - ttft)
                tpot = decode_ms / max(1, tok_count - 1)
                metrics.append(RequestMetric(ttft, e2e, tok_count, tpot, True))
            except Exception:
                t_end = time.perf_counter()
                metrics.append(RequestMetric(0.0, (t_end - t0) * 1000.0, 0, 0.0, False))

    wall_start = time.perf_counter()
    await asyncio.gather(*(_worker(p) for p in prompts))
    wall_sec = max(1e-6, time.perf_counter() - wall_start)

    ok = [m for m in metrics if m.success]
    total_tokens = sum(m.output_tokens for m in ok)
    return BenchmarkSummary(
        total_requests=len(prompts),
        concurrency=concurrency,
        success_rate=round(len(ok) / max(1, len(prompts)), 4),
        p50_ttft_ms=percentile([m.ttft_ms for m in ok], 0.50),
        p95_ttft_ms=percentile([m.ttft_ms for m in ok], 0.95),
        avg_tpot_ms=round(sum(m.tpot_ms for m in ok) / max(1, len(ok)), 2),
        throughput_tokens_per_sec=round(total_tokens / wall_sec, 2),
        total_wall_sec=round(wall_sec, 3),
    )


if __name__ == "__main__":
    test_prompts = [f"请解释星辰智脑工单处理规范第{i}条" for i in range(15)]
    summary = asyncio.run(run_benchmark(test_prompts, concurrency=5))
    print("压测报告:", summary)


### 🔍 步骤 2 代码深度解析

1. 使用 `asyncio.Semaphore(concurrency)` 精确控制并发水位。
2. `stream_fn` 采用依赖注入设计，本地跑单测用 `mock_vllm_stream`，生产压测只需传入基于 `httpx.AsyncClient.stream()` 请求真实 vLLM/Ollama 的函数即可零改动复用。

**预期运行输出参考：**
```text
压测报告: BenchmarkSummary(total_requests=15, concurrency=5, success_rate=1.0, p50_ttft_ms=22.4, p95_ttft_ms=26.8, avg_tpot_ms=4.3, throughput_tokens_per_sec=920.5, total_wall_sec=0.391)
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day55/tests/test_day55.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 55 单元测试：PagedAttention 模拟与异步推理压测工具"""
import asyncio
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day55_inference_benchmark import percentile, run_benchmark
from src.day55_paged_attention_sim import simulate_kv_cache_allocation


def test_paged_attention_dramatically_reduces_waste():
    lens = [200, 350, 600, 150, 820, 400]
    cmp = simulate_kv_cache_allocation(lens, max_model_len=4096, block_size=16)
    assert cmp.naive_waste_ratio > 0.80
    assert cmp.paged_waste_ratio < 0.05
    assert cmp.memory_saved_ratio > 0.85


def test_percentile_calculation():
    assert percentile([10.0, 20.0, 30.0, 40.0, 50.0], 0.50) == 30.0
    assert percentile([10.0, 20.0, 30.0, 40.0, 50.0], 0.95) == 50.0


def test_async_benchmark_runs_and_reports_metrics():
    prompts = [f"测试问题 {i}" for i in range(8)]
    report = asyncio.run(run_benchmark(prompts, concurrency=4))
    assert report.total_requests == 8
    assert report.success_rate == 1.0
    assert report.p50_ttft_ms > 0
    assert report.p95_ttft_ms >= report.p50_ttft_ms
    assert report.throughput_tokens_per_sec > 0

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：为压测报告增加 SLA 自动达标判定函数

**题目要求：**
编写 `check_sla(summary, max_p95_ttft_ms=1500.0, min_tps=40.0, min_success_rate=0.99)`，返回 `(passed: bool, failed_reasons: list[str])`。

**思路提示：** 逐项比对 `summary.p95_ttft_ms`、`summary.throughput_tokens_per_sec` 与 `summary.success_rate`。


In [ ]:
from src.day55_inference_benchmark import BenchmarkSummary


def check_sla(
    summary: BenchmarkSummary,
    max_p95_ttft_ms: float = 1500.0,
    min_tps: float = 40.0,
    min_success_rate: float = 0.99,
):
    reasons = []
    if summary.p95_ttft_ms > max_p95_ttft_ms:
        reasons.append(f"P95 TTFT({summary.p95_ttft_ms}ms) 超出 SLA({max_p95_ttft_ms}ms)")
    if summary.throughput_tokens_per_sec < min_tps:
        reasons.append(f"吞吐({summary.throughput_tokens_per_sec}) 低于 SLA({min_tps})")
    if summary.success_rate < min_success_rate:
        reasons.append(f"成功率({summary.success_rate}) 低于 SLA({min_success_rate})")
    return len(reasons) == 0, reasons


demo = BenchmarkSummary(10, 5, 1.0, 20.0, 35.0, 4.0, 320.0, 0.5)
print(check_sla(demo))


**答案解析与评分标准：**

将 SLA 判定自动化，可直接嵌入 GitLab CI 发布流水线，性能回退自动阻断上线。


### 作业 2（企业实战）：企业实战：编写面向真实 Ollama / vLLM 的 OpenAI 兼容 SSE 流式适配器

**题目要求：**
使用 `httpx.AsyncClient` 编写 `openai_compat_stream(base_url, api_key, model, prompt)` 异步生成器，解析 `data: {...}` SSE 行并逐块 `yield` 增量文本。

**思路提示：** 遍历 `async for line in resp.aiter_lines():`，跳过空行和 `data: [DONE]`，解析 `choices[0]['delta'].get('content')`。


In [ ]:
import json
from typing import AsyncIterator
import httpx


async def openai_compat_stream(
    base_url: str, api_key: str, model: str, prompt: str
) -> AsyncIterator[str]:
    url = f"{base_url.rstrip('/')}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}"}
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True,
    }
    async with httpx.AsyncClient(timeout=60.0) as client:
        async with client.stream("POST", url, headers=headers, json=payload) as resp:
            resp.raise_for_status()
            async for line in resp.aiter_lines():
                if not line.startswith("data: "):
                    continue
                data = line[6:].strip()
                if data == "[DONE]":
                    break
                chunk = json.loads(data)
                delta = chunk["choices"][0]["delta"].get("content", "")
                if delta:
                    yield delta


print("OpenAI 兼容流式适配器定义完成，可直接注入 run_benchmark！")


**答案解析与评分标准：**

标准 SSE 解析器同时兼容 Ollama (`http://localhost:11434/v1`)、vLLM (`http://localhost:8000/v1`)、DeepSeek 与通义千问。


### 作业 3（架构进阶）：架构进阶：模拟 Prefix Caching 在多轮 RAG 对话中的显存与 Prefill 加速收益

**题目要求：**
假设 10 个并发请求共享 800 tokens 的长 System Prompt，各自携带 200~500 tokens 的用户问题。计算开启 Prefix Caching 前后总 Prefill 计算量下降百分比。

**思路提示：** 关闭时总 Prefill = `sum(shared + u for u in user_lens)`；开启后共享前缀仅首条计算一次，总 Prefill = `shared + sum(user_lens)`。


In [ ]:
def calc_prefix_cache_gain(shared_prompt_tokens: int, user_token_lens: list[int]) -> float:
    without_cache = sum(shared_prompt_tokens + u for u in user_token_lens)
    with_cache = shared_prompt_tokens + sum(user_token_lens)
    return round((without_cache - with_cache) / without_cache * 100, 2)


gain = calc_prefix_cache_gain(800, [220, 310, 450, 280, 390, 260, 500, 330, 290, 410])
print(f"开启 Prefix Caching 后 Prefill 计算量直降: {gain}%")


**答案解析与评分标准：**

对于固定长系统指令或高频命中的热门文档，Prefix Caching 可削减 60%+ 的重复 Prefill 计算，大幅降低 TTFT。


---

## 🎯 老陈收尾复盘

漂亮！今天这套压测工具不仅能测本地模拟网关，改一下 `base_url` 明天就能直接打到公司真实 vLLM 集群上出 P50/P95/P99 首字延迟和 TPS 报告。记住面试官最爱问的三个推理指标：TTFT、TPOT、Throughput，今天全用代码吃透了！
